In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/shubhangig870@gmail.com/sephoraa/1_Setup/Utility

In [0]:
dbutils.widgets.text("catalog","sephoraa","catalog")
dbutils.widgets.text("data_source","customer_addresses","data_source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
display(df_bronze)

In [0]:
df_bronze.printSchema()

In [0]:
df_bronze.columns

In [0]:
#address_id

from pyspark.sql.functions import col, count, when

new=df_bronze.filter(col("address_id").rlike("^ADDR"))
display(new)

In [0]:
#customer_id
new = df_bronze.filter(col("customer_id").rlike("^CUST"))
display(new)

In [0]:
# Address_line1
new = df_bronze.withColumn("Address_line1",when(col("Address_line1").cast("string")=="N/A","unknown").otherwise(col("Address_line1")))
display(new)

new = df_bronze.withColumn("Address_line1",when(col("Address_line1").isNull(),"unknown").otherwise(col("Address_line1")))
display(new)

new = df_bronze.groupBy("Address_line1").count().filter(col("count")>1)
display(new)

new = df_bronze.filter(col("Address_line1").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
display(new)

#new = df_bronze.fillna({"Address_line1": "UNKNOWN"})



In [0]:
#city
new = df_bronze.groupBy("city").count().filter(col("count")>1)
display(new)

new = df_bronze.withColumn("city",when(col("city").isNull(),"unknown").otherwise(col("city")))
display(new)



In [0]:
#country
new = df_bronze.groupBy("city").count().filter(col("count")>1)
display(new)

# column mdhil all letter small letter mdhe convert krte #
from pyspark.sql.functions import lower, col

new = df_bronze.withColumn(
    "country",
    lower(col("country"))
)

display(new)


# column mdhil capital letter and small letter right krte #
from pyspark.sql.functions import col, trim, initcap, lower, when

new = df_bronze.withColumn(
    "country",
    when(
        lower(trim(col("country"))).isin(
            "nan", "n/a", "#n/a",
            "none", "null", "<null>",
            "@@@err", "###corrupted###",
            "[binary_data]", "\\x00\\x00", "???"
        ),
        None
    ).otherwise(
        initcap(lower(trim(col("country"))))
    )
)

display(new)


In [0]:
# postal_code
new = df_bronze.withColumn("postal_code",when(col("postal_code").isNull(),"unknown").otherwise(col("postal_code")))
display(new)

new = df_bronze.withColumn("postal_code",when(col("postal_code").cast("string")=="???","unknown").otherwise(col("postal_code")))
display(new)



In [0]:
 # address_type
 new = df_bronze.withColumn("address_type",when(col("address_type").cast("string")=="N/A","unknown").otherwise(col("address_type")))
display(new)

new = df_bronze.withColumn("address_type",when(col("address_type").isNull(),"unknown").otherwise(col("address_type")))
display(new)

new = df_bronze.groupBy("address_type").count().filter(col("count")>1)
display(new)

new = df_bronze.filter(col("Address_line1").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
display(new)


In [0]:
#is_default

In [0]:
# created_at
from pyspark.sql.functions import col,when
from pyspark.sql import functions as F
new = df_bronze.withColumn(
    "created_at",
    F.coalesce(
        # Date-only formats
        F.try_to_date(F.trim(F.col("created_at")), F.lit("yyyy/MM/dd")),
        F.try_to_date(F.trim(F.col("created_at")), F.lit("dd/MM/yyyy")),
        F.try_to_date(F.trim(F.col("created_at")), F.lit("yyyy-MM-dd")),
        F.try_to_date(F.trim(F.col("created_at")), F.lit("dd-MM-yyyy")),
        # Timestamp formats
        F.try_to_timestamp(F.trim(F.col("created_at")), F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp(F.trim(F.col("created_at")), F.lit("yyyy/MM/dd HH:mm:ss"))
    )
)
new = df_bronze.withColumn("created_at", F.to_date("created_at"))
display(new)

from pyspark.sql.functions import col, when, current_date

new = df_bronze.withColumn(
    "created_at",
    when(
        col("created_at").isNull(),
        current_date()
    ).otherwise(col("created_at"))
)

display(new)

In [0]:
# updated_at
from pyspark.sql.functions import col,when
from pyspark.sql import functions as F
new = df_bronze.withColumn(
    "updated_at",
    F.coalesce(
        # Date-only formats
        F.try_to_date(F.trim(F.col("updated_at")), F.lit("yyyy/MM/dd")),
        F.try_to_date(F.trim(F.col("updated_at")), F.lit("dd/MM/yyyy")),
        F.try_to_date(F.trim(F.col("updated_at")), F.lit("yyyy-MM-dd")),
        F.try_to_date(F.trim(F.col("updated_at")), F.lit("dd-MM-yyyy")),
        # Timestamp formats
        F.try_to_timestamp(F.trim(F.col("updated_at")), F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp(F.trim(F.col("updated_at")), F.lit("yyyy/MM/dd HH:mm:ss"))
    )
)
new = df_bronze.withColumn("updated_at", F.to_date("updated_at"))
 #display(new)

from pyspark.sql.functions import col, when, current_date

new = df_bronze.withColumn(
    "updated_at",
    when(
        col("updated_at").isNull(),
        current_date()
    ).otherwise(col("updated_at"))
)

display(new)

In [0]:
# ingestion_date

from pyspark.sql.functions import col,when
from pyspark.sql import functions as F
new = df_bronze.withColumn(
    "ingestion_date",
    F.coalesce(
        # Date-only formats
        F.try_to_date(F.trim(F.col("ingestion_date")), F.lit("yyyy/MM/dd")),
        F.try_to_date(F.trim(F.col("ingestion_date")), F.lit("dd/MM/yyyy")),
        F.try_to_date(F.trim(F.col("ingestion_date")), F.lit("yyyy-MM-dd")),
        F.try_to_date(F.trim(F.col("ingestion_date")), F.lit("dd-MM-yyyy")),
        # Timestamp formats
        F.try_to_timestamp(F.trim(F.col("ingestion_date")), F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp(F.trim(F.col("ingestion_date")), F.lit("yyyy/MM/dd HH:mm:ss"))
    )
)
new = df_bronze.withColumn("ingestion_date", F.to_date("ingestion_date"))
# display(df_silver)

from pyspark.sql.functions import col, when, current_date

new = df_bronze.withColumn(
    "ingestion_date",
    when(
        col("ingestion_date").isNull(),
        current_date()
    ).otherwise(col("ingestion_date"))
)

display(new)

In [0]:

 


 
 'is_default',
 
